# RAG Trust Scoring — Step 1–5: experimental base & attack generation

Runs on **free Colab** (Runtime → Change runtime type → **T4 GPU**).

| Step | Script | What it produces | Time on T4 |
|---|---|---|---|
| 1 | `step1_prepare_data.py` | BEIR-NQ corpus + PoisonedRAG target questions | ~10 min (download) |
| 2 | `step2_generate_attacks.py` | 7 attack families × 100 questions × 5 passages | seconds |
| 3 | `step3_retrieve.py` | Contriever retrieval with the poisoned knowledge base | ~2 min |
| 4 | `step4_run_rag.py` | Vanilla-RAG baseline: ASR / accuracy per attack | ~15–25 min (HF) |
| 5 | `step5_build_scorer_dataset.py` | labelled passage dataset for the trust scorer | seconds |

Everything is saved to Google Drive, so a disconnect does not lose finished steps.

## 0. Setup — Drive, code, dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/rag_trust_project'   # all data/results live here
import os
os.makedirs(WORK, exist_ok=True)
%cd $WORK
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU - steps 1-3 and 5 still work on CPU"

Upload **`rag_trust.zip`** (the file Claude gave you) into `MyDrive/rag_trust_project/` once, or run the cell below and pick it from your computer.

In [ ]:
if not os.path.exists(f'{WORK}/rag_trust/common.py'):
    if not os.path.exists(f'{WORK}/rag_trust.zip'):
        from google.colab import files
        files.upload()                      # choose rag_trust.zip
    !unzip -oq rag_trust.zip -d $WORK
!ls rag_trust

In [ ]:
# Experimental base: PoisonedRAG (USENIX Security'25). We reuse its pre-computed Contriever
# retrieval results and the GPT-4-generated poisoned texts - no need to run its old-torch code.
!git clone --depth 1 https://github.com/sleeepeer/PoisonedRAG.git 2>/dev/null || echo "already cloned"
# Reference only (templates were adapted into rag_trust/attack_templates.py):
!git clone --depth 1 https://github.com/WeifeiJin/CleanBase.git 2>/dev/null || echo "already cloned"
!git clone --depth 1 https://github.com/liu00222/Open-Prompt-Injection.git 2>/dev/null || echo "already cloned"
!pip -q install openai accelerate

## Step 1 — Prepare BEIR-NQ + target questions

In [ ]:
!python rag_trust/step1_prepare_data.py --dataset nq --poisonedrag_dir PoisonedRAG --beir_dir /content/datasets --out_dir data

## Step 2 — Generate adversarial passages (7 families)

In [ ]:
!python rag_trust/step2_generate_attacks.py --dataset nq --out_dir data --adv_per_query 5

## Step 3 — Retrieval with the poisoned knowledge base (Contriever)
Check the `[sanity]` line: the recomputed score should match PoisonedRAG's stored score (difference ≈ 0).

In [ ]:
!python rag_trust/step3_retrieve.py --dataset nq --out_dir data --encoder contriever --top_k 5

## Step 4 — Vanilla RAG baseline (no defense)

Pick ONE generator:
* **HF (free, GPU)**: `--backend hf` → Qwen2.5-3B-Instruct on the T4.
* **OpenAI API**: `--backend openai --model gpt-4o-mini` — add your key in Colab *Secrets* (🔑 icon) as `OPENAI_API_KEY` first.

Do a 10-question trial first, then the full run.

In [ ]:
BACKEND = 'hf'          # or 'openai'
MODEL   = ''            # '' = default (Qwen/Qwen2.5-3B-Instruct for hf, gpt-4o-mini for openai)

if BACKEND == 'openai':
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
model_arg = f'--model {MODEL}' if MODEL else ''
!python rag_trust/step4_run_rag.py --dataset nq --backend $BACKEND $model_arg --defense none --max_questions 10 --results_dir results_trial

In [ ]:
# full run: no defense (lower bound) and oracle filter (upper bound)
!python rag_trust/step4_run_rag.py --dataset nq --backend $BACKEND $model_arg --defense none
!python rag_trust/step4_run_rag.py --dataset nq --backend $BACKEND $model_arg --defense oracle

In [ ]:
import pandas as pd, glob
for f in sorted(glob.glob('results/nq/*/summary.csv')):
    print('\n==', f); display(pd.read_csv(f))

## Step 5 — Passage-level dataset for the trust scorer

In [ ]:
!python rag_trust/step5_build_scorer_dataset.py --dataset nq --out_dir data
import pandas as pd
df = pd.read_json('data/nq/scorer_dataset.jsonl', lines=True)
display(pd.crosstab(df.family, df.split))

### Look at a few failures
Which retrieved passages made the LLM output the wrong answer?

In [ ]:
import json
rows = [json.loads(l) for l in open('results/nq/none_' + BACKEND + ('-' + MODEL.split('/')[-1] if MODEL else '') + '/ignore.jsonl')]
for r in [x for x in rows if x['attack_success']][:3]:
    print('Q:', r['question'], '| correct:', r['correct'], '| target:', r['incorrect'])
    print('LLM:', r['response'][:200]); print('-'*80)

## Step 6 — Trust scorer (signals → 5-fold CV → scored pools)

**Prompt Guard 2 is a gated model.** To use it: open https://huggingface.co/meta-llama/Llama-Prompt-Guard-2-22M,
accept the licence, create a read token (HF → Settings → Access Tokens) and add it to Colab *Secrets* as `HF_TOKEN`.
Without it the script automatically falls back to `protectai/deberta-v3-base-prompt-injection-v2` (ungated) —
just write down which one was used (it is printed and saved in `trust/latency.json`).

In [ ]:
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded')
except Exception:
    print('no HF_TOKEN secret - will fall back to the ungated protectai classifier')
!pip -q install datasets
!python rag_trust/step6_trust_scorer.py --dataset nq --data_dir data --target_fpr 0.05

In [ ]:
import pandas as pd
det = pd.read_csv('data/nq/trust/detection_metrics.csv')
display(det.pivot(index='variant', columns='setting', values='AUROC').round(3))
display(det.pivot(index='variant', columns='setting', values='TPR'))
display(pd.read_csv('data/nq/trust/feature_weights.csv'))
display(det.pivot(index='variant', columns='setting', values='precision'))
if os.path.exists('data/nq/trust/overdefense.csv'):
    print('Over-defense on benign instruction-like passages (NotInject):')
    display(pd.read_csv('data/nq/trust/overdefense.csv'))
print(open('data/nq/trust/latency.json').read()[:300])

## Step 4 again — end-to-end ASR / accuracy with each trust variant

In [ ]:
for d in ['trust_rule', 'trust_guard', 'trust_ctx', 'trust_fused']:
    !python rag_trust/step4_run_rag.py --dataset nq --backend $BACKEND $model_arg --defense $d

In [ ]:
# main results table: ASR per attack (rows) x defense (columns); last rows = clean accuracy and FPR budget
import glob
tabs = {}
for f in sorted(glob.glob('results/nq/*/summary.csv')):
    name = f.split('/')[-2].rsplit('_', 1)[0] if f.split('/')[-2].endswith('_hf') else f.split('/')[-2]
    tabs[name] = pd.read_csv(f).set_index('setting')
order = [c for c in ['none', 'trust_rule', 'trust_guard', 'trust_ctx', 'trust_fused', 'oracle'] if c in tabs]
asr = pd.DataFrame({c: tabs[c]['ASR'] for c in order})
asr.loc['clean Acc (no attack)'] = [tabs[c].loc['none', 'Acc'] for c in order]
asr.loc['kept_clean (no attack)'] = [tabs[c].loc['none', 'kept_clean'] for c in order]
display(asr)
asr.to_csv('results/nq/main_table.csv')

## Step 7 — Statistics (95% CIs + paired McNemar tests)
With n = 100 questions, a difference of a few points may be noise. This gives Wilson confidence
intervals for every number and an exact McNemar test between `trust_fused` and every other defense.

In [ ]:
!python rag_trust/step7_stats.py --dataset nq --results_dir results --suffix _hf
display(pd.read_csv('results/nq/stats_mcnemar.csv'))

## Experiment 2 — Single adversarial passage per question (where does the method fail?)
The context signals may rely on the attacker injecting **5 near-duplicate passages**. Here every
question gets only **1** adversarial passage (cf. AuthChain, one-document poisoning). Same pipeline,
separate folders (`data_single/`, `results_single/`), nothing from the main experiment is overwritten.

In [ ]:
!mkdir -p data_single/nq && cp data/nq/clean_retrieval.json data/nq/targets.json data_single/nq/
!python rag_trust/step2_generate_attacks.py --dataset nq --out_dir data_single --adv_per_query 1
!python rag_trust/step3_retrieve.py --dataset nq --out_dir data_single
!python rag_trust/step6_trust_scorer.py --dataset nq --data_dir data_single

In [ ]:
for d in ['none', 'trust_rule', 'trust_guard', 'trust_ctx', 'trust_fused', 'oracle']:
    !python rag_trust/step4_run_rag.py --dataset nq --data_dir data_single --results_dir results_single --backend $BACKEND $model_arg --defense $d
!python rag_trust/step7_stats.py --dataset nq --results_dir results_single --suffix _hf

In [ ]:
def main_table(root):
    tabs = {}
    for f in sorted(glob.glob(f'{root}/nq/*/summary.csv')):
        tabs[f.split('/')[-2].rsplit('_', 1)[0]] = pd.read_csv(f).set_index('setting')
    order = [c for c in ['none', 'trust_rule', 'trust_guard', 'trust_ctx', 'trust_fused', 'oracle'] if c in tabs]
    t = pd.DataFrame({c: tabs[c]['ASR'] for c in order})
    t.loc['clean Acc (no attack)'] = [tabs[c].loc['none', 'Acc'] for c in order]
    t.loc['adv@k (avg over attacks)'] = [round(tabs[c].drop('none')['adv@k'].mean(), 2) for c in order]
    return t
single = main_table('results_single'); display(single); single.to_csv('results_single/nq/main_table.csv')